# 01 Combined Experiments and Controls

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`run_combined.py`](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/scripts/run_combined.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/logs)
- [`run_controls.py`](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/scripts/run_controls.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/logs)
- [`review_evidence.py`](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/scripts/review_evidence.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/Extra_Trees/Combined/scripts/run_combined.py': {'ref': 'ec30f7fd4a97de9f8336df2eef6a52b394c810c0', 'sha': '65a0ec0cfd4a393829fd5906c4097aa205eb862d'}, '05_model_optimize/Extra_Trees/Combined/scripts/run_controls.py': {'ref': 'ec30f7fd4a97de9f8336df2eef6a52b394c810c0', 'sha': '0da643c4fefc145501c4c345906e115c5e6cc207'}, '05_model_optimize/Extra_Trees/Combined/scripts/review_evidence.py': {'ref': 'ec30f7fd4a97de9f8336df2eef6a52b394c810c0', 'sha': '610dd37f4d044d8c4036ec6258f3c763b4d27516'}}

## run_combined

원본: [05_model_optimize/Extra_Trees/Combined/scripts/run_combined.py](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/scripts/run_combined.py)  
Git blob: `65a0ec0cfd4a393829fd5906c4097aa205eb862d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/logs)

Run the shared Extra Trees experiment runner in Combined mode.



In [ ]:
%%archive_source 05_model_optimize/Extra_Trees/Combined/scripts/run_combined.py
"""Run the shared Extra Trees experiment runner in Combined mode."""
import runpy
import sys
from pathlib import Path


if __name__ == "__main__":
    sys.argv[1:1] = ["--mode", "combined"]
    runner = Path(__file__).resolve().parents[2] / "Normal/scripts/run_extra_trees.py"
    runpy.run_path(str(runner), run_name="__main__")


## run_controls

원본: [05_model_optimize/Extra_Trees/Combined/scripts/run_controls.py](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/scripts/run_controls.py)  
Git blob: `0da643c4fefc145501c4c345906e115c5e6cc207`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/logs)

Run paired Combined Extra Trees development controls on the archived folds.

CLI 인자 (정적 추출): `--arm`, `--out`, `--jobs`

In [ ]:
%%archive_source 05_model_optimize/Extra_Trees/Combined/scripts/run_controls.py
"""Run paired Combined Extra Trees development controls on the archived folds."""
import argparse
from dataclasses import replace
import json
from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "Normal" / "scripts"))
from run_extra_trees import Config, Study

SEEDS = (42, 17, 73)
ARMS = {
    "stroke00": {"prep": "size"},
    "features08": {"max_features": 0.08},
}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--arm", choices=ARMS, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--jobs", type=int, default=4)
    args = parser.parse_args()
    args.mode = "combined"
    args.out.mkdir(parents=True, exist_ok=True)
    selection = Path(__file__).resolve().parents[1] / "results" / "selection.json"
    candidate = Config(**json.loads(selection.read_text())["config"])
    config = replace(candidate, **ARMS[args.arm])
    study = Study(args)
    study.audit()
    for seed in SEEDS:
        study.run(f"C{seed}_{args.arm}", config, seed=seed)


if __name__ == "__main__":
    main()


## review_evidence

원본: [05_model_optimize/Extra_Trees/Combined/scripts/review_evidence.py](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/scripts/review_evidence.py)  
Git blob: `610dd37f4d044d8c4036ec6258f3c763b4d27516`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Combined/logs)

Verify archived predictions and produce paired Extra Trees development diagnostics.

CLI 인자 (정적 추출): `--out`, `--controls`

In [ ]:
%%archive_source 05_model_optimize/Extra_Trees/Combined/scripts/review_evidence.py
"""Verify archived predictions and produce paired Extra Trees development diagnostics."""
import argparse
import csv
import json
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BASE = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(BASE / "Normal" / "scripts"))
from run_extra_trees import DATA, LABELS, WRITERS, load_official, load_writers
from run_extra_trees import official_average, score, write_csv


def read_summary(folder):
    return json.loads((folder / "summary.json").read_text())


def verify_scores(expected, actual):
    for key in ("n", "correct", "accuracy", "macro_f1", "macro_precision",
                "macro_recall", "confusion_matrix", "focused_errors"):
        if expected[key] != actual[key]:
            raise ValueError(f"Prediction and summary disagree for {key}")


def verify_runs(folders, official, writers, table_path=None):
    labels = np.concatenate([writers[w]["y"] for w in WRITERS])
    table = {}
    if table_path is not None:
        with table_path.open(newline="") as stream:
            table = {row["run"]: row for row in csv.DictReader(stream)}
        if set(table) != {folder.name for folder in folders}:
            raise ValueError(f"CSV run list differs from archive: {table_path}")
    for folder in folders:
        summary = read_summary(folder)
        combined = summary["training_source"] == "combined"
        folds = {str(writer) for writer in WRITERS} if combined else {"all"}
        if set(summary["official"]) != folds or set(summary["timing"]) != folds:
            raise ValueError(f"Unexpected folds: {folder}")
        for fold in folds:
            planned_writers = [writer for writer in WRITERS if writer != int(fold)] if combined else []
            records = summary["timing"][fold]
            if records["training_writers"] != planned_writers or (
                records["handmade_train_n"] != sum(len(writers[w]["y"]) for w in planned_writers)
            ):
                raise ValueError(f"Writer holdout mismatch: {folder}, {fold}")
        with np.load(folder / "predictions.npz", allow_pickle=False) as predictions:
            for writer in WRITERS:
                key = str(writer)
                verify_scores(summary["writers"][key],
                              score(writers[writer]["y"], predictions[f"writer_{key}"]))
            pooled = np.concatenate([predictions[f"writer_{w}"] for w in WRITERS])
            verify_scores(summary["hand"], score(labels, pooled))
            for fold, metrics in summary["official"].items():
                verify_scores(metrics, score(official["y_val"], predictions[f"official_{fold}"]))
        if table:
            row = table[folder.name]
            values = {"official_accuracy": official_average(summary),
                      "official_macro_f1": official_average(summary, "macro_f1"),
                      "hand_accuracy": summary["hand"]["accuracy"],
                      "hand_macro_f1": summary["hand"]["macro_f1"]}
            if any(not np.isclose(value, float(row[key]), rtol=0, atol=1e-12)
                   for key, value in values.items()):
                raise ValueError(f"CSV scores differ from predictions: {folder}")


def error_diagnostics(out, writers):
    directional_rows, changed_examples = [], []
    for mode in ("Normal", "Combined"):
        runs = BASE / mode / "results" / "runs"
        with np.load(runs / "P00_raw" / "predictions.npz") as old, np.load(
            runs / "R42_selected" / "predictions.npz"
        ) as new:
            for writer in WRITERS:
                y = writers[writer]["y"]
                previous, current = (run[f"writer_{writer}"] for run in (old, new))
                is_seven = y == 7
                before = is_seven & (previous == 9)
                after = is_seven & (current == 9)
                directional_rows.append(dict(
                    mode=mode, writer=writer, true_sevens=int(is_seven.sum()),
                    correct_before=int((is_seven & (previous == 7)).sum()),
                    correct_after=int((is_seven & (current == 7)).sum()),
                    before=int(before.sum()), after=int(after.sum()),
                    persistent=int((before & after).sum()),
                    resolved_to_correct=int((before & (current == 7)).sum()),
                    resolved_to_other_error=int((before & ~after & (current != 7)).sum()),
                    introduced_from_correct=int((after & (previous == 7)).sum()),
                    introduced_from_other_error=int((after & ~before & (previous != 7)).sum()),
                ))
                for index in np.flatnonzero(before != after):
                    changed_examples.append(dict(
                        comparison=f"{mode}_baseline_to_selected", writer=writer, index=int(index),
                        actual="7", before=LABELS[previous[index]], after=LABELS[current[index]],
                    ))

    runs = BASE / "Combined" / "results" / "runs"
    y = writers[41]["y"]
    class_rows = []
    with np.load(runs / "CTRL_normal" / "predictions.npz") as old, np.load(
        runs / "R42_selected" / "predictions.npz"
    ) as new:
        previous, current = old["writer_41"], new["writer_41"]
        for i, label in enumerate(LABELS):
            in_class = y == i
            before_correct, after_correct = in_class & (previous == y), in_class & (current == y)
            class_rows.append(dict(
                actual=label, n=int(in_class.sum()),
                errors_before=int(in_class.sum() - before_correct.sum()),
                errors_after=int(in_class.sum() - after_correct.sum()),
                corrected=int((after_correct & ~before_correct).sum()),
                regressed=int((before_correct & ~after_correct).sum()),
            ))
        for index in np.flatnonzero((previous == y) != (current == y)):
            changed_examples.append(dict(
                comparison="writer41_official_only_to_combined", writer=41, index=int(index),
                actual=LABELS[y[index]], before=LABELS[previous[index]], after=LABELS[current[index]],
            ))
    write_csv(out / "seven_to_nine_by_writer.csv", directional_rows)
    write_csv(out / "writer41_by_class.csv", class_rows)
    write_csv(out / "paired_error_examples.csv", changed_examples)
    for mode in ("Normal", "Combined"):
        seven_gallery(out, mode, changed_examples, writers)


def seven_gallery(out, mode, examples, writers):
    source = [row for row in examples
              if row["comparison"] == f"{mode}_baseline_to_selected"]
    new_errors = sorted((row for row in source if row["after"] == "9"),
                        key=lambda row: (row["before"] != "7", row["writer"], row["index"]))
    corrected = [row for row in source if row["before"] == "9" and row["after"] == "7"]
    panels = new_errors[:4] + corrected[:4]
    fig, axes = plt.subplots(1, len(panels), figsize=(10, 1.6))
    for ax, row in zip(axes, panels):
        image = writers[row["writer"]]["X"][row["index"]].reshape(28, 28)
        ax.imshow(image, cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{row['writer']} / {row['index']}\n{row['before']} → {row['after']}", fontsize=9)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(out / f"seven_to_nine_{mode.lower()}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)


def summarize_controls(out):
    archive = BASE / "Combined" / "results" / "runs"
    rows = []
    for arm in ("stroke00", "features08"):
        for seed in (42, 17, 73):
            reference = read_summary(archive / f"R{seed}_selected")
            control = read_summary(out / "runs" / f"C{seed}_{arm}")
            expected_key = {"stroke00": "prep", "features08": "max_features"}[arm]
            differences = {key for key, value in reference["config"].items()
                           if value != control["config"][key]}
            if differences != {expected_key} or reference["seed"] != control["seed"]:
                raise ValueError(f"Unmatched controls: {arm}, {seed}")
            for name, run in ((arm, control), ("T02_reference", reference)):
                rows.append(dict(
                    comparison=arm, arm=name, seed=seed,
                    official_accuracy=official_average(run), hand_accuracy=run["hand"]["accuracy"],
                    hand_macro_f1=run["hand"]["macro_f1"], hand_7_to_9=run["hand"]["focused_errors"]["7->9"],
                    writer41_accuracy=run["writers"]["41"]["accuracy"],
                    fit_seconds_mean=float(np.mean([f["fit_seconds"] for f in run["timing"].values()])),
                ))
    write_csv(out / "paired_controls.csv", rows)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--controls", action="store_true")
    args = parser.parse_args()
    args.out.mkdir(parents=True, exist_ok=True)
    official = load_official(DATA / "datasets/MNIST_EMNIST_official_split_112000.npz")
    writers = load_writers(DATA / "Hand_made")
    total = 0
    for mode in ("Normal", "Combined"):
        results = BASE / mode / "results"
        folders = sorted((results / "runs").iterdir())
        verify_runs(folders, official, writers, results / "experiment_summary.csv")
        total += len(folders)
    error_diagnostics(args.out, writers)
    if args.controls:
        folders = sorted((args.out / "runs").iterdir())
        verify_runs(folders, official, writers)
        summarize_controls(args.out)
        total += len(folders)
    print(f"Verified {total} archived prediction and summary pairs.", flush=True)


if __name__ == "__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")